# BanglaPhonologyBench — M5 Zero-Shot Evaluation (open models)

Runs the frozen task datasets (G2P, syllable count, rhyme awareness/generation,
schwa deletion) as zero-shot prompts against an open model and scores the
completions with `scripts/zeroshot_lib.py`. See `docs/DEVELOPMENT_LOG.md`
(M5 section) and `CLAUDE.md` for background — this is the first M5 run,
**TigerLLM-9B-it only** (cost-free; more open models and later closed models
come after this one is validated).

## Why row-level resumability this time (unlike M4)

M4's extraction was one forward pass per word — fast enough that file-level
"skip if the whole combo is already done" was enough (see M4 notebook's own
reasoning). Zero-shot **generation** is autoregressive and far slower per
item, and a full-dataset run across all 5 tasks is thousands of generate()
calls — a real multi-hour, real restart-risk job. So generation here saves
**one JSONL row per item** as it goes (`checkpoints/zeroshot/{task}_{lang}.jsonl`)
and skips ids already present on restart. Scoring is a separate, cheap,
re-runnable step over whatever's been generated so far.


## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" sentencepiece


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/LihanCanCode/BanglaPhonologyBench.git"
REPO_DIR = "/kaggle/working/BanglaPhonologyBench"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import zeroshot_lib as zs
from tokenizer_adapter import TOKENIZER_SPECS

print("repo ready:", REPO_DIR)


In [ ]:
# HF token: prefer Kaggle Secrets (Add-ons -> Secrets -> add HF_TOKEN). Not
# needed for TigerLLM (ungated) but harmless to set up now for later models.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("HF_TOKEN set:", bool(HF_TOKEN))


## 2. Checkpoint restore

Set `CHECKPOINT_INPUT` to a previous run's attached output dataset once you
have one. Leave as-is for a first run.

In [ ]:
CKPT_DIR = "/kaggle/working/checkpoints/zeroshot"
os.makedirs(CKPT_DIR, exist_ok=True)

CHECKPOINT_INPUT = "/kaggle/input/banglaphonologybench-m5-checkpoints"

if os.path.isdir(CHECKPOINT_INPUT):
    subprocess.run(f"cp -r {CHECKPOINT_INPUT}/. {CKPT_DIR}/", shell=True, check=False)
    print(f"restored checkpoints from {CHECKPOINT_INPUT}")
else:
    print("no checkpoint input attached -- starting fresh (expected on a first run)")


## 3. Config

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_KEY = "tigerllm"          # only model for this first M5 pass
TASKS_DIR = os.path.join(REPO_DIR, "data", "tasks")

# Bangla-language prompts first (the spec's primary condition); add "en" to
# LANGS once a bn-only pass has been validated -- doubles the generate()
# calls, so it's opt-in rather than default here.
LANGS = ["bn"]

# None = full dataset. Set an int (e.g. 300) for a fast first smoke-test
# pass -- rerun with None later to fill in the rest (already-generated ids
# are skipped, this just changes how many NEW ids get added this session).
SAMPLE_SIZE = 300

RHYME_GEN_K = 5


## 4. Load model

TigerLLM-9B-it needs 4-bit quantization to fit a T4 (same reasoning as
M4's probing notebook for the 8-9B models). Uses the tokenizer's chat
template if it has one (instruction-tuned models expect that format).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_generate_fn(llm_key, max_new_tokens=64):
    repo_candidates = TOKENIZER_SPECS[llm_key]
    last_err = None
    for repo_id in repo_candidates:
        try:
            tok = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            model = AutoModelForCausalLM.from_pretrained(
                repo_id, token=HF_TOKEN, quantization_config=bnb_config,
                device_map="auto",
            )
            model.eval()

            def generate_fn(prompt, _tok=tok, _model=model):
                # return_dict=True -> a proper BatchEncoding (input_ids +
                # attention_mask) unpacked as **kwargs into generate(), not
                # a bare tensor passed positionally -- passing a bare
                # tensor there is fragile across transformers versions
                # (some builds choke trying to read .shape off whatever
                # apply_chat_template happened to return).
                if _tok.chat_template:
                    messages = [{"role": "user", "content": prompt}]
                    encoded = _tok.apply_chat_template(
                        messages, add_generation_prompt=True,
                        return_tensors="pt", return_dict=True,
                    ).to(_model.device)
                else:
                    encoded = _tok(prompt, return_tensors="pt").to(_model.device)
                with torch.no_grad():
                    out = _model.generate(
                        **encoded, max_new_tokens=max_new_tokens, do_sample=False,
                        pad_token_id=_tok.eos_token_id,
                    )
                new_tokens = out[0][encoded["input_ids"].shape[1]:]
                return _tok.decode(new_tokens, skip_special_tokens=True).strip()

            return generate_fn
        except Exception as e:
            last_err = e
            print(f"  failed to load {repo_id}: {e}")
    raise RuntimeError(f"could not load any repo for {llm_key}: {last_err}")


print(f"loading {MODEL_KEY}...")
generate_fn = load_generate_fn(MODEL_KEY)
print("model loaded")


## 5. Generation (row-level resumable, one task per cell)

Reads a task's dataset, builds prompts via `zeroshot_lib`, generates a
completion per row, appends `{id, raw_output}` to a JSONL checkpoint file —
skipping ids already present. Safe to interrupt and re-run.

**Split into one cell per task on purpose** (not a single loop over all 5):
run a task's cell, then inspect its output (section 6's scoring cell works
on whatever's been generated so far, or just eyeball the raw JSONL) before
spending GPU time on the next task. Catches a bad prompt/parser on 300
items instead of after committing hours to the full run.

In [ ]:
import json


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_done_ids(path):
    if not os.path.exists(path):
        return set()
    with open(path, encoding="utf-8") as f:
        return {json.loads(line)["id"] for line in f if line.strip()}


TASK_SOURCES = {
    "g2p": (os.path.join(TASKS_DIR, "g2p.jsonl"), zs.prompt_g2p, lambda r: r["orth"]),
    "syllable_count": (os.path.join(TASKS_DIR, "syllable_count_word.jsonl"),
                        zs.prompt_syllable_count, lambda r: r["orth"]),
    "rhyme_awareness": (os.path.join(REPO_DIR, "data", "task3a_rhyme_pairs.jsonl"),
                         lambda r, lang: zs.prompt_rhyme_awareness(r["orth1"], r["orth2"], lang), None),
    "rhyme_generation": (os.path.join(REPO_DIR, "data", "task3b_rhyme_generation.jsonl"),
                          lambda r, lang: zs.prompt_rhyme_generation(r["prompt_word"], RHYME_GEN_K, lang), None),
    "schwa_deletion": (os.path.join(TASKS_DIR, "schwa_deletion.jsonl"),
                        lambda r, lang: zs.prompt_schwa(
                            r["orth"], [r["grapheme_clusters"][p] for p in r["schwa_positions"]], lang),
                        None),
}


def generate_for_task(task, lang):
    path, prompt_fn, _ = TASK_SOURCES[task]
    rows = load_jsonl(path)
    if SAMPLE_SIZE is not None:
        rows = rows[:SAMPLE_SIZE]

    out_path = os.path.join(CKPT_DIR, f"{task}_{lang}_{MODEL_KEY}.jsonl")
    done = load_done_ids(out_path)
    todo = [r for r in rows if r["id"] not in done]
    print(f"{task}/{lang}: {len(done)} already done, {len(todo)} to generate")

    with open(out_path, "a", encoding="utf-8") as f:
        for i, r in enumerate(todo):
            prompt = prompt_fn(r, lang)
            raw = generate_fn(prompt)
            f.write(json.dumps({"id": r["id"], "raw_output": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (i + 1) % 25 == 0:
                print(f"  {task}/{lang}: {i + 1}/{len(todo)}")
    print(f"{task}/{lang}: done ({out_path})")


### 5a. G2P

In [ ]:
for lang in LANGS:
    generate_for_task("g2p", lang)


### 5b. Syllable count

In [ ]:
for lang in LANGS:
    generate_for_task("syllable_count", lang)


### 5c. Rhyme awareness (Task 3a)

In [ ]:
for lang in LANGS:
    generate_for_task("rhyme_awareness", lang)


### 5d. Rhyme generation (Task 3b)

In [ ]:
for lang in LANGS:
    generate_for_task("rhyme_generation", lang)


### 5e. Schwa deletion

In [ ]:
for lang in LANGS:
    generate_for_task("schwa_deletion", lang)


### Quick peek at any task's raw output

Run after any of the 5a-5e cells above to eyeball what the model actually
said before moving on to the next task.

In [ ]:
def peek(task, lang="bn", n=8):
    path = os.path.join(CKPT_DIR, f"{task}_{lang}_{MODEL_KEY}.jsonl")
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(json.loads(line))

peek("g2p")  # change the task name to check a different one


## 6. Scoring

Re-reads whatever's been generated (doesn't need the model loaded) and
applies `zeroshot_lib`'s parsers/scorers. Safe to re-run any time, including
on a partial run -- just scores however many rows exist so far.

In [ ]:
import csv

SCORE_FNS = {
    "g2p": zs.run_g2p,
    "syllable_count": zs.run_syllable_count,
    "rhyme_awareness": zs.run_rhyme_awareness,
    "rhyme_generation": zs.run_rhyme_generation,
    "schwa_deletion": zs.run_schwa,
}


def score_task(task, lang):
    path, _, _ = TASK_SOURCES[task]
    rows = load_jsonl(path)
    if SAMPLE_SIZE is not None:
        rows = rows[:SAMPLE_SIZE]

    ckpt_path = os.path.join(CKPT_DIR, f"{task}_{lang}_{MODEL_KEY}.jsonl")
    if not os.path.exists(ckpt_path):
        return None
    raw_by_id = {json.loads(l)["id"]: json.loads(l)["raw_output"]
                 for l in open(ckpt_path, encoding="utf-8") if l.strip()}
    scored_rows = [r for r in rows if r["id"] in raw_by_id]
    if not scored_rows:
        return None

    def canned_generate_fn(_prompt, _row_iter=iter(scored_rows)):
        return raw_by_id[next(_row_iter)["id"]]

    # replay in the same order as scored_rows so canned_generate_fn's
    # internal iterator lines up with each task runner's internal loop
    return SCORE_FNS[task](scored_rows, canned_generate_fn, lang)


results = []
for task in TASK_SOURCES:
    for lang in LANGS:
        s = score_task(task, lang)
        if s is None:
            print(f"{task}/{lang}: nothing generated yet")
            continue
        results.append(s)
        print(f"{task:18s} {lang}  n={s.n:5d} parsed={s.n_parsed:5d}  {s.metrics}")

out_csv = os.path.join(CKPT_DIR, f"zeroshot_summary_{MODEL_KEY}.csv")
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["task", "lang", "n", "n_parsed", "metric", "value"])
    for s in results:
        for k, v in s.metrics.items():
            w.writerow([s.task, s.lang, s.n, s.n_parsed, k, v])
print("wrote", out_csv)


## 7. Save your progress

**Click "Save Version"** so `/kaggle/working/checkpoints/` persists. Next
session, attach that output as an input dataset, set `CHECKPOINT_INPUT`
above to match, and re-run — already-generated ids are skipped
automatically, and `SAMPLE_SIZE` can be raised (or set to `None`) to fill
in the rest without redoing what's already there.